# 4. Advanced Feature Engineering: Solving Feature Starvation

## 4.1 Overview and Objectives

In Notebook 3, our XGBoost baseline achieved a 70.60% accuracy but suffered from a high False Positive rate when predicting switches. The Feature Importance analysis revealed that the model over-relies on the identity of "Pivot" Pokémon (e.g., Corviknight, Alomomola) because it lacks the necessary contextual field data to make nuanced decisions.

The objective of this notebook is to resolve this **Feature Starvation**. We will re-unroll the dataset and apply advanced Regular Expressions and Window Functions to extract a dynamic, turn-by-turn state representation, including:
1. **Health Point (HP) Tracking:** Parsing `|-damage|`, `|-heal|`, and `|switch|` protocols to track the active Pokémon's remaining HP.
2. **Hazard Tracking:** Identifying the presence of Stealth Rock to provide the AI with entry hazard context.

In [1]:
import os
import polars as pl

# ==============================================================================
# --- 1. SET ENVIRONMENT PATHS & INITIALIZE LAZYFRAME ---
# ==============================================================================
print("Initializing Polars LazyFrame and Unrolling text logs...")

PROJECT_ROOT = os.path.abspath("../../../")
CACHE_DIR = os.path.join(PROJECT_ROOT, "data/models/dataset/huggingface_cache")
PARQUET_PATH = os.path.join(
    CACHE_DIR,
    "hub/datasets--milkkarten--pokechamp/snapshots/b5820ff5d0c8d5e5cec692d55f756b6e5a66a203/data/train-*.parquet"
)

# 1. Scan and Filter
lf = pl.scan_parquet(PARQUET_PATH)
expert_lf = lf.filter((pl.col("gamemode") == "gen9randombattle") & (pl.col("elo") == "1800+"))

# 2. Unroll the Text Logs (Explode by turn)
turn_lf = expert_lf.with_columns([
    pl.col("text").str.split("\n|turn|").alias("turns_list")
]).explode("turns_list")

turn_lf = turn_lf.with_columns([
    pl.col("turns_list").str.extract(r"^(\d+)").cast(pl.Int32, strict=False).alias("turn_number")
]).filter(pl.col("turn_number").is_not_null())

# Let's grab the first battle again to test our new Regex on!
first_battle_id = "2211658410-2024-09-28"
test_df = turn_lf.filter(pl.col("battle_id") == first_battle_id).select([
    "turn_number", "turns_list"
]).head(5).collect()

print("Dataset successfully unrolled! Ready for Advanced Regex.")
display(test_df)

Initializing Polars LazyFrame and Unrolling text logs...
Dataset successfully unrolled! Ready for Advanced Regex.


turn_number,turns_list
i32,str
1,"""1 | |t:|1727547025 |-end|p2a: …"
2,"""2 | |t:|1727547032 |-end|p2a: …"
3,"""3 | |t:|1727547036 |switch|p1a…"
4,"""4 | |t:|1727547040 |switch|p2a…"
5,"""5 | |t:|1727547052 |-end|p2a: …"


## 4.2 Dynamic HP Tracking and State Memory

A critical component of the "Feature Starvation" identified in Notebook 3 is the model's inability to quantify risk. A human expert will switch a "Pivot" Pokémon if it is at 10% HP, but might leave it in if it is at 100% HP.

To map this continuous state, we extract the remaining Health Points (HP) for both players.
* **Greedy Regex Matching:** Because a Pokémon can take multiple instances of damage in a single turn (e.g., hazard damage followed by direct attack damage), we utilize a greedy regex (`.*`) to ensure we capture only the final HP state broadcasted before the turn ends.
* **Windowed Imputation:** We apply `.forward_fill()` over the `battle_id` to maintain the HP memory across stagnant turns. For initial turns where no HP broadcast has occurred yet, we impute a value of `1.0` (100% HP).

In [2]:
# ==============================================================================
# --- 2. EXTRACTING CONTINUOUS HP MEMORY ---
# ==============================================================================
print("Extracting Turn-by-Turn HP Percentages...")

# 1. Extract the Raw HP Numbers
turn_lf = turn_lf.with_columns([
    # The greedy '.*' skips to the LAST time p1a/p2a is mentioned in the turn log.
    # It captures the numerator (Current HP) or '0' if the string is "0 fnt".
    pl.col("turns_list").str.extract(r".*p1a: [^|]+\|([0-9]+)").cast(pl.Float64).alias("p1_hp_current"),
    pl.col("turns_list").str.extract(r".*p2a: [^|]+\|([0-9]+)").cast(pl.Float64).alias("p2_hp_current"),

    # It captures the denominator (Max HP). If fainted, this might be null.
    pl.col("turns_list").str.extract(r".*p1a: [^|]+\|[0-9]+/([0-9]+)").cast(pl.Float64).alias("p1_hp_max"),
    pl.col("turns_list").str.extract(r".*p2a: [^|]+\|[0-9]+/([0-9]+)").cast(pl.Float64).alias("p2_hp_max")
])

# 2. Apply the "Memory" (Forward Fill across the Battle)
turn_lf = turn_lf.with_columns([
    pl.col("p1_hp_current").forward_fill().over("battle_id"),
    pl.col("p2_hp_current").forward_fill().over("battle_id"),
    pl.col("p1_hp_max").forward_fill().over("battle_id"),
    pl.col("p2_hp_max").forward_fill().over("battle_id")
])

# 3. Calculate the actual HP Percentage (0.0 to 1.0)
turn_lf = turn_lf.with_columns([
    # We divide current by max. If max is null (due to fainting), we fill with 100 to avoid division errors.
    # If the resulting percentage is still null (e.g., Turn 1 before any damage), we assume 100% health (1.0).
    (pl.col("p1_hp_current") / pl.col("p1_hp_max").fill_null(100.0)).fill_null(1.0).alias("p1_hp_percent"),
    (pl.col("p2_hp_current") / pl.col("p2_hp_max").fill_null(100.0)).fill_null(1.0).alias("p2_hp_percent")
])

# Let's view the results for the first 5 turns to validate the memory!
print("\nHP Tracking Results:")
hp_df = turn_lf.filter(pl.col("battle_id") == first_battle_id).select([
    "turn_number", "p1_hp_percent", "p2_hp_percent"
]).head(5).collect()

display(hp_df)

Extracting Turn-by-Turn HP Percentages...

HP Tracking Results:


turn_number,p1_hp_percent,p2_hp_percent
i32,f64,f64
1,1.0,1.0
2,1.0,0.66
3,1.0,0.66
4,1.0,0.97
5,1.0,0.97


## 4.3 Macro-State Tracking: Hazards and Terastallization

Beyond direct HP, human experts base their switching decisions on the macro-state of the field. Two of the most defining mechanics of the Generation 9 OU metagame are Entry Hazards (Stealth Rock) and Terastallization.

To encode these states, we must transition from simple presence detection to **temporal cumulative tracking**:
* **Stealth Rock State:** We identify the `|-sidestart|` protocol (hazard set) as `+1` and the `|-sideend|` protocol (hazard removed) as `-1`. By applying a cumulative sum over the `battle_id`, we create a persistent boolean flag indicating whether hazards are currently active on a player's side of the field.
* **Tera State:** Terastallization is a once-per-game mechanic. We extract the `|-terastallize|` protocol and apply a cumulative sum. A value of `1` indicates the player has burned their Tera, fundamentally altering the risk calculations for the remainder of the match.

In [4]:
# ==============================================================================
# --- 3. EXTRACTING HAZARDS & TERA WITH CUMULATIVE SUMS ---
# ==============================================================================
print("Extracting Stealth Rock and Tera States...")

turn_lf = turn_lf.with_columns([
    # --- STEALTH ROCK TRACKING ---
    # When P2 sets rocks, it goes on P1's side (-sidestart|p1)
    pl.col("turns_list").str.contains(r"\|-sidestart\|p1:[^|]+\|move: Stealth Rock").cast(pl.Int32).alias("p1_sr_set"),
    pl.col("turns_list").str.contains(r"\|-sideend\|p1:[^|]+\|move: Stealth Rock").cast(pl.Int32).alias("p1_sr_removed"),

    pl.col("turns_list").str.contains(r"\|-sidestart\|p2:[^|]+\|move: Stealth Rock").cast(pl.Int32).alias("p2_sr_set"),
    pl.col("turns_list").str.contains(r"\|-sideend\|p2:[^|]+\|move: Stealth Rock").cast(pl.Int32).alias("p2_sr_removed"),

    # --- TERA TRACKING ---
    pl.col("turns_list").str.contains(r"\|-terastallize\|p1a:").cast(pl.Int32).alias("p1_used_tera_this_turn"),
    pl.col("turns_list").str.contains(r"\|-terastallize\|p2a:").cast(pl.Int32).alias("p2_used_tera_this_turn")
])

# Calculate the Cumulative States over the course of the battle
turn_lf = turn_lf.with_columns([
    # Stealth Rock is Active if (Sets - Removes) > 0
    (pl.col("p1_sr_set") - pl.col("p1_sr_removed")).cum_sum().over("battle_id").clip(lower_bound=0, upper_bound=1).alias("p1_stealth_rock_active"),
    (pl.col("p2_sr_set") - pl.col("p2_sr_removed")).cum_sum().over("battle_id").clip(lower_bound=0, upper_bound=1).alias("p2_stealth_rock_active"),

    # Has the player used Tera yet this game? (1 = Yes, 0 = No)
    pl.col("p1_used_tera_this_turn").cum_sum().over("battle_id").clip(lower_bound=0, upper_bound=1).alias("p1_tera_used"),
    pl.col("p2_used_tera_this_turn").cum_sum().over("battle_id").clip(lower_bound=0, upper_bound=1).alias("p2_tera_used")
])

# Let's check a specific battle where we know Tera and Rocks happened to see the cumulative math in action!
# (Using head(15) so we can see the progression deeper into a match)
print("\nMacro-State Tracking Results:")
macro_df = turn_lf.filter(pl.col("battle_id") == first_battle_id).select([
    "turn_number", "p1_stealth_rock_active", "p1_tera_used", "p2_tera_used"
]).head(30).collect()

display(macro_df)

Extracting Stealth Rock and Tera States...

Macro-State Tracking Results:


turn_number,p1_stealth_rock_active,p1_tera_used,p2_tera_used
i32,i32,i32,i32
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,0,0,0
…,…,…,…
18,0,0,0
19,0,0,0
20,0,1,0


## 4.4 Matrix Assembly and Export

With the advanced contextual features extracted (HP, Hazards, Terastallization), we must merge them with our foundational State-Action matrix generated in Notebook 2.

By executing an **Inner Join** on the primary keys (`battle_id` and `turn_number`), we combine the categorical field state (Active Pokémon) with the continuous risk metrics (HP) and macro-states (Hazards/Tera). Finally, we drop the unstructured text and export the ultimate, context-rich dataset to disk for the final XGBoost training phase.

In [8]:
# ==============================================================================
# --- 4. JOINING THE MATRICES & EXPORTING ---
# ==============================================================================
print("Assembling the Ultimate ML Matrix...")

# 1. Isolate the new advanced features (Dropping the heavy 'turns_list' text)
advanced_lf = turn_lf.select([
    "battle_id", "turn_number",
    "p1_hp_percent", "p2_hp_percent",
    "p1_stealth_rock_active", "p2_stealth_rock_active",
    "p1_tera_used", "p2_tera_used"
])

# 2. Load the base matrix we created back in Notebook 2
base_path = os.path.join(PROJECT_ROOT, "data/models/dataset/processed/expert_gen9randombattle_unrolled.parquet")
base_lf = pl.scan_parquet(base_path)

# 3. Perform the Inner Join!
# This matches the new HP/Hazard data to the exact exact battle and turn where the expert made a Move/Switch.
ultimate_lf = base_lf.join(advanced_lf, on=["battle_id", "turn_number"], how="inner")

# 4. Export the final dataset
output_path = os.path.join(PROJECT_ROOT, "data/models/dataset/processed/expert_gen9randombattle_advanced.parquet")
print(f"Streaming merged data to: {output_path}")

# sink_parquet executes the entire out-of-core pipeline and writes to disk
ultimate_lf.sink_parquet(output_path)

print("Export complete! The Advanced Dataset is ready.")

# 5. Let's load it back into memory to admire the final structure!
final_advanced_df = pl.read_parquet(output_path)
print(f"\nFinal Advanced Matrix Shape: {final_advanced_df.shape[0]:,} rows and {final_advanced_df.shape[1]} columns")
display(final_advanced_df.head(5))

Assembling the Ultimate ML Matrix...
Streaming merged data to: /home/gerardpf/TFM/data/models/dataset/processed/expert_gen9randombattle_advanced.parquet
Export complete! The Advanced Dataset is ready.

Final Advanced Matrix Shape: 402,132 rows and 11 columns


battle_id,turn_number,y_p1_action,p1_active_pokemon,p2_active_pokemon,p1_hp_percent,p2_hp_percent,p1_stealth_rock_active,p2_stealth_rock_active,p1_tera_used,p2_tera_used
str,i32,i32,str,str,f64,f64,i32,i32,i32,i32
"""2195493737-2024-09-05""",26,0,"""dragonite""","""dragapult""",0.78,0.41,0,0,1,1
"""2211601226-2024-09-28""",10,1,"""dragapult""","""lokix""",0.0,0.88,0,0,1,1
"""2211942870-2024-09-28""",26,0,"""kingambit""","""latios""",0.0,0.79,0,0,0,1
"""2211942870-2024-09-28""",31,1,"""iron moth""","""great tusk""",0.0,0.73,0,0,1,1
"""2201193377-2024-09-13""",5,0,"""sandaconda""","""iron treads""",0.15,0.59,0,0,0,0


## 4.5 Conclusion: The Contextual Matrix is Ready

In this phase, we successfully solved the "Feature Starvation" problem identified in the initial XGBoost baseline. By applying greedy Regular Expressions and Window Functions (Cumulative Sums and Forward Fills) to the raw Showdown text logs, we extracted a dynamic, temporal representation of the battlefield.

**Key Achievements:**
1. **Continuous Risk Tracking:** Successfully quantified the exact HP percentages for both players across all turns, providing the model with a vital measure of survival probability.
2. **Macro-State Tracking:** Encoded the presence of Stealth Rock and the depletion of Terastallization as persistent boolean flags.
3. **Matrix Assembly:** Executed a high-performance inner join to merge these advanced metrics with the base categorical matrix.

**Next Steps:**
We will transition to the final modeling phase. We will load this newly generated `expert_gen9randombattle_advanced.parquet` dataset, re-apply the `GroupShuffleSplit` and One-Hot Encoding, and train a new XGBoost agent to see if this contextual awareness resolves the False Positive bottleneck and pushes our ROC AUC to state-of-the-art levels.